# model4.0.ipynb — XGBoost + LSTM on Structured Assay-Level Dataset

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from src.models.tree_models import get_xgboost_model
from src.models.lstm import AssayLSTM, AssayDataset, train_lstm_model
from src.utils.visualization import plot_training_curve, plot_actual_vs_predicted
from src.utils.metrics import print_evaluation_metrics

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
df = pd.read_csv('../data/assay_dataset.csv', parse_dates=['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)
TARGET_COLS = ['pct_Iron_Concentrate', 'pct_Silica_Concentrate']
DROP_COLS   = ['datetime'] + TARGET_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

split_idx = int(len(df) * 0.8)
X_raw = df[feature_cols].copy()
y = df[TARGET_COLS].values.astype(np.float32)

X_train_raw, X_test_raw = X_raw.iloc[:split_idx], X_raw.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw).astype(np.float32)
X_test_imp = imputer.transform(X_test_raw).astype(np.float32)

feat_scaler = StandardScaler()
tgt_scaler = StandardScaler()
X_train_sc = feat_scaler.fit_transform(X_train_imp).astype(np.float32)
X_test_sc = feat_scaler.transform(X_test_imp).astype(np.float32)
y_train_sc = tgt_scaler.fit_transform(y_train).astype(np.float32)
y_test_sc = tgt_scaler.transform(y_test).astype(np.float32)

## 1. XGBoost

In [ ]:
xgb_models, xgb_preds = {}, {}
y_pred_xgb_matrix = np.zeros_like(y_test)
for i, target in enumerate(TARGET_COLS):
    m = get_xgboost_model()
    m.fit(X_train_imp, y_train[:, i])
    pred = m.predict(X_test_imp)
    xgb_models[target] = m
    y_pred_xgb_matrix[:, i] = pred

print('XGBoost Evaluation:')
print_evaluation_metrics(y_test, y_pred_xgb_matrix, TARGET_COLS)

## 2. LSTM

In [ ]:
SEQ_LEN = 20
def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len : i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

X_tr_seq, y_tr_seq = make_sequences(X_train_sc, y_train_sc, SEQ_LEN)
X_te_seq, y_te_seq = make_sequences(X_test_sc,  y_test_sc,  SEQ_LEN)

train_loader = DataLoader(AssayDataset(X_tr_seq, y_tr_seq), batch_size=64, shuffle=False)
test_loader = DataLoader(AssayDataset(X_te_seq, y_te_seq), batch_size=64, shuffle=False)

model = AssayLSTM(n_features=X_tr_seq.shape[2]).to(DEVICE)
model, train_losses, val_losses = train_lstm_model(model, train_loader, test_loader, epochs=200, device=DEVICE)
plot_training_curve(train_losses, val_losses, 'AssayLSTM Training Curve')

In [ ]:
model.eval()
preds_sc = []
with torch.no_grad():
    for Xb, _ in test_loader:
        preds_sc.append(model(Xb.to(DEVICE)).cpu().numpy())

lstm_pred_sc = np.vstack(preds_sc)
lstm_pred = tgt_scaler.inverse_transform(lstm_pred_sc)
lstm_true = tgt_scaler.inverse_transform(y_te_seq)

print('LSTM Evaluation:')
print_evaluation_metrics(lstm_true, lstm_pred, TARGET_COLS)